## **Dataset**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Reproduzir exatamente a base da aula
np.random.seed(42)

filmes = [
    'Star Wars', 'Matrix', 'Interestelar',
    'Titanic', 'Diário de uma Paixão',
    'O Poderoso Chefão', 'Pulp Fiction',
    'Toy Story', 'Procurando Nemo', 'O Iluminado'
]

perfis_genero = {
    'Ficção Científica': ['Star Wars', 'Matrix', 'Interestelar'],
    'Romance':           ['Titanic', 'Diário de uma Paixão'],
    'Drama':             ['O Poderoso Chefão', 'Pulp Fiction'],
    'Animação':          ['Toy Story', 'Procurando Nemo'],
    'Terror':            ['O Iluminado']
}

generos = list(perfis_genero.keys())
usuarios = [f'U{i:03d}' for i in range(1, 51)]

matriz_ratings = pd.DataFrame(np.nan, index=usuarios, columns=filmes)

for user_id in usuarios:
    genero_fav = np.random.choice(generos)
    genero_sec = np.random.choice([g for g in generos if g != genero_fav])
    n_avaliacoes = np.random.randint(3, 9)
    filmes_avaliados = np.random.choice(filmes, size=n_avaliacoes, replace=False)
    for filme in filmes_avaliados:
        if filme in perfis_genero[genero_fav]:
            nota = np.random.choice([4, 5], p=[0.4, 0.6])
        elif filme in perfis_genero[genero_sec]:
            nota = np.random.choice([3, 4, 5], p=[0.3, 0.4, 0.3])
        else:
            nota = np.random.choice([1, 2, 3, 4], p=[0.2, 0.3, 0.3, 0.2])
        matriz_ratings.loc[user_id, filme] = nota

In [2]:
print(matriz_ratings.head())
print(f"\nDensidade: {matriz_ratings.notna().mean().mean():.2%}")
print(f"Total de avaliações: {matriz_ratings.notna().sum().sum()}")

      Star Wars  Matrix  Interestelar  Titanic  Diário de uma Paixão  \
U001        3.0     NaN           NaN      1.0                   NaN   
U002        NaN     NaN           2.0      NaN                   NaN   
U003        2.0     2.0           NaN      3.0                   NaN   
U004        NaN     3.0           NaN      4.0                   1.0   
U005        3.0     4.0           1.0      4.0                   5.0   

      O Poderoso Chefão  Pulp Fiction  Toy Story  Procurando Nemo  O Iluminado  
U001                4.0           NaN        NaN              5.0          3.0  
U002                1.0           3.0        4.0              1.0          3.0  
U003                5.0           4.0        2.0              NaN          NaN  
U004                NaN           4.0        NaN              5.0          4.0  
U005                3.0           NaN        4.0              NaN          1.0  

Densidade: 55.00%
Total de avaliações: 275


##  **Alternating Least Squares (ALS)**

In [3]:
class ALSFromScratch:

    def __init__(self, n_fatores=5, n_iteracoes=30, lambda_reg=0.1,
                 random_state=42, clip=(1, 5)):
        self.n_fatores = n_fatores
        self.n_iteracoes = n_iteracoes
        self.lambda_reg = lambda_reg
        self.random_state = random_state
        self.clip = clip
        self.historico_erro = []

    def fit(self, matriz):
        rng = np.random.default_rng(self.random_state)
        n_usuarios, n_itens = matriz.shape

        # Máscara de valores observados
        mask = ~np.isnan(matriz.values)
        R = np.nan_to_num(matriz.values)  # ausentes viram 0 (não usados)

        # Inicialização dos fatores latentes
        X = rng.normal(0, 0.1, (n_usuarios, self.n_fatores))
        Y = rng.normal(0, 0.1, (n_itens, self.n_fatores))

        #Cria a matriz identidade
        I_k = np.eye(self.n_fatores)

        for it in range(self.n_iteracoes):
            # --- Passo 1: fixar Y, resolver X (por usuário) ---
            for u in range(n_usuarios):
                idx = np.where(mask[u])[0]
                if len(idx) == 0:
                    # Usuário sem avaliações: mantém inicialização aleatória
                    continue
                Y_u = Y[idx]
                r_u = R[u, idx]
                A = Y_u.T @ Y_u + self.lambda_reg * I_k
                b = Y_u.T @ r_u
                X[u] = np.linalg.solve(A, b)

            # --- Passo 2: fixar X, resolver Y (por item) ---
            for i in range(n_itens):
                idx = np.where(mask[:, i])[0]
                if len(idx) == 0:
                    # Item sem avaliações: mantém inicialização aleatória
                    continue
                X_i = X[idx]
                r_i = R[idx, i]
                A = X_i.T @ X_i + self.lambda_reg * I_k
                b = X_i.T @ r_i
                Y[i] = np.linalg.solve(A, b)

            # --- Erro de treino (RMSE sobre observados, com clipping) ---
            R_hat = X @ Y.T
            if self.clip is not None:
                R_hat = np.clip(R_hat, *self.clip)
            erro = np.sqrt(np.mean((R_hat[mask] - R[mask]) ** 2))
            self.historico_erro.append(erro)

        self.X_ = X
        self.Y_ = Y
        self.mask_ = mask
        self.R_ = R
        return self

    def prever(self):
        """Retorna a matriz completa de previsões (com clipping)."""
        R_hat = self.X_ @ self.Y_.T
        if self.clip is not None:
            R_hat = np.clip(R_hat, *self.clip)
        return R_hat

    def prever_para(self, usuario_idx, item_idx):
        """Previsão para um par (usuário, item) específico."""
        val = self.X_[usuario_idx] @ self.Y_[item_idx]
        print(val)
        if self.clip is not None:
            val = np.clip(val, *self.clip)
        return val

## **Treinamento e avaliação**

In [4]:
from sklearn.model_selection import train_test_split

mask_obs = matriz_ratings.notna().values
idx_obs = np.argwhere(mask_obs)
idx_treino, idx_teste = train_test_split(idx_obs, test_size=0.2, random_state=42)

matriz_treino = matriz_ratings.copy()
for u, i in idx_teste:
    matriz_treino.iloc[u, i] = np.nan

# ------------------------------------------------------------
# Validação: garantir que ninguém ficou sem treino
# ------------------------------------------------------------
assert not matriz_treino.isna().all(axis=1).any(), \
    "Há usuário(s) sem nenhuma avaliação no treino!"
assert not matriz_treino.isna().all(axis=0).any(), \
    "Há item(ns) sem nenhuma avaliação no treino!"

print("Split validado: todos os usuários e itens têm ao menos 1 avaliação no treino.")
print(f"Avaliações no treino: {matriz_treino.notna().sum().sum()}")
print(f"Avaliações no teste:  {len(idx_teste)}")

Split validado: todos os usuários e itens têm ao menos 1 avaliação no treino.
Avaliações no treino: 220
Avaliações no teste:  55


## **Treianmento**

In [5]:
# Treinar
als = ALSFromScratch(n_fatores=5, n_iteracoes=25,
                     lambda_reg=0.1, random_state=42,
                     clip=(1, 5))
als.fit(matriz_treino)

# Avaliar no teste (com clipping já aplicado em prever())
R_hat = als.prever()
erros = []
for u, i in idx_teste:
    real = matriz_ratings.iloc[u, i]
    prev = R_hat[u, i]
    erros.append((real - prev) ** 2)

rmse_teste = np.sqrt(np.mean(erros))
mae_teste = np.mean(np.abs([
    matriz_ratings.iloc[u, i] - R_hat[u, i] for u, i in idx_teste
]))

print(f"RMSE treino (última iteração): {als.historico_erro[-1]:.4f}")
print(f"RMSE teste:                    {rmse_teste:.4f}")
print(f"MAE  teste:                    {mae_teste:.4f}")

RMSE treino (última iteração): 0.1037
RMSE teste:                    2.2065
MAE  teste:                    1.7611


In [6]:
def recomendar_para_usuario(modelo, matriz_treino, usuario, top_n=3):
    u_idx = matriz_treino.index.get_loc(usuario)
    print(u_idx)
    scores = modelo.prever()[u_idx]

    # Filtrar apenas itens NÃO avaliados no treino
    nao_avaliados = matriz_treino.loc[usuario].isna().values
    candidatos = np.where(nao_avaliados)[0]

    ranking = candidatos[np.argsort(-scores[candidatos])][:top_n]
    return [(matriz_treino.columns[i], round(scores[i], 3)) for i in ranking]

In [7]:
for u in ['U001']:
    print(f"\n{u} — recomendações:")
    for filme, score in recomendar_para_usuario(als, matriz_treino, u, top_n=3):
        print(f"  {filme:<25} score={score}")


U001 — recomendações:
0
  Pulp Fiction              score=2.038
  Matrix                    score=1.0
  Interestelar              score=1.0


## **Aplicando métricas de avaliação**

* **Candidatos** = {itens NÃO avaliados no treino}
           = {itens que podem ser recomendados legitimamente}

* **Relevantes** = {itens do TESTE com nota ≥ 3}

## **Implementação das métricas**

In [11]:
import numpy as np
import pandas as pd

def precision_at_k(recomendados, relevantes, k):
    top_k = recomendados[:k]
    return sum(1 for i in top_k if i in relevantes) / k if k else 0.0

def recall_at_k(recomendados, relevantes, k):
    if not relevantes: return 0.0
    top_k = recomendados[:k]
    return sum(1 for i in top_k if i in relevantes) / len(relevantes)

def average_precision_at_k(recomendados, relevantes, k):
    if not relevantes: return 0.0
    top_k, hits, soma = recomendados[:k], 0, 0.0
    for pos, item in enumerate(top_k, start=1):
        if item in relevantes:
            hits += 1
            soma += hits / pos
    return soma / min(len(relevantes), k)

def ndcg_at_k(recomendados, relevantes, k):
    top_k = recomendados[:k]
    dcg = sum(1.0 / np.log2(pos + 1)
              for pos, item in enumerate(top_k, start=1)
              if item in relevantes)
    n_rel = min(len(relevantes), k)
    idcg = sum(1.0 / np.log2(pos + 1) for pos in range(1, n_rel + 1))
    return dcg / idcg if idcg > 0 else 0.0

##  **Avaliação em lote**

In [9]:
def avaliar_ranking_teste(modelo, matriz_treino, matriz_teste,
                          k=5, threshold=3.0):

    R_hat = modelo.prever()
    precisions, recalls, aps, ndcgs = [], [], [], []

    for u_idx, usuario in enumerate(matriz_treino.index):
        aval_teste = matriz_teste.loc[usuario]

        # Candidatos = itens com nota OBSERVADA no teste
        candidatos = np.array([
            matriz_teste.columns.get_loc(c)
            for c in aval_teste.index
            if pd.notna(aval_teste[c])
        ])

        if len(candidatos) == 0:
            continue

        # Relevantes = notas >= threshold
        relevantes = {
            i for i in candidatos
            if matriz_teste.iloc[u_idx, i] >= threshold
        }
        if not relevantes:
            continue

        scores = R_hat[u_idx, candidatos]
        recomendados = candidatos[np.argsort(-scores)].tolist()

        precisions.append(precision_at_k(recomendados, relevantes, k))
        recalls.append(recall_at_k(recomendados, relevantes, k))
        aps.append(average_precision_at_k(recomendados, relevantes, k))
        ndcgs.append(ndcg_at_k(recomendados, relevantes, k))

    return {
        f"Precision@{k}": np.mean(precisions),
        f"Recall@{k}":    np.mean(recalls),
        f"MAP@{k}":       np.mean(aps),
        f"NDCG@{k}":      np.mean(ndcgs),
        "n_usuarios":     len(precisions),
    }

## **Executando**

In [10]:
for k in [3, 5]:
    print(f"\n=== Ranking @ K={k} ===")
    for nome, valor in avaliar_ranking_teste(
        als, matriz_treino, matriz_ratings, k=k
    ).items():
        print(f"  {nome}: {valor:.4f}" if isinstance(valor, float)
              else f"  {nome}: {valor}")


=== Ranking @ K=3 ===
  Precision@3: 0.8600
  Recall@3: 0.7561
  MAP@3: 0.9311
  NDCG@3: 0.9464
  n_usuarios: 50

=== Ranking @ K=5 ===
  Precision@5: 0.6800
  Recall@5: 0.9321
  MAP@5: 0.9138
  NDCG@5: 0.9428
  n_usuarios: 50
